# AWS Bedrock Debug Test

Testing Claude Sonnet & Haiku 4.5 exactly like the working test notebook to debug agent issues.


In [1]:
%pip install requests python-dotenv

import json
import os
from dotenv import load_dotenv
import requests

load_dotenv()



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


True

In [2]:
# AWS Configuration - EXACTLY like test notebook
aws_bearer_token = os.getenv("AWS_BEARER_TOKEN_BEDROCK") or os.getenv("AWS_BEARER_TOKEN")
aws_region = os.getenv("AWS_REGION", "us-east-1")

# Model IDs for Bedrock - Claude 4.5
sonnet_model_id = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
haiku_model_id = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

if not aws_bearer_token:
    print("❌ AWS Bearer Token not found")
else:
    print(f"✅ AWS Bearer Token found")
    print(f"Region: {aws_region}")
    print(f"Sonnet Model: {sonnet_model_id}")
    print(f"Haiku Model: {haiku_model_id}")


✅ AWS Bearer Token found
Region: us-west-2
Sonnet Model: us.anthropic.claude-sonnet-4-5-20250929-v1:0
Haiku Model: us.anthropic.claude-haiku-4-5-20251001-v1:0


In [3]:
# Using requests library with bearer token for Bedrock API - EXACTLY like test notebook
import requests

# Bedrock converse API endpoint - EXACTLY like test notebook
bedrock_endpoint = f"https://bedrock-runtime.{aws_region}.amazonaws.com"

if aws_bearer_token:
    print("✅ Ready to use Bedrock API with bearer token")
    print(f"Endpoint: {bedrock_endpoint}")
else:
    print("⚠️  Bearer token not found")


✅ Ready to use Bedrock API with bearer token
Endpoint: https://bedrock-runtime.us-west-2.amazonaws.com


In [4]:
# Helper function to call Bedrock converse API - EXACTLY like test notebook
def call_bedrock_converse(model_id, prompt, max_tokens=300, temperature=0.7):
    """Call Bedrock converse API using bearer token authentication."""
    if not aws_bearer_token:
        return None, "Bearer token not found"
    
    url = f"{bedrock_endpoint}/model/{model_id}/converse"
    
    headers = {
        "Authorization": f"Bearer {aws_bearer_token}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "messages": [
            {
                "role": "user",
                "content": [{"text": prompt}]
            }
        ],
        "inferenceConfig": {
            "maxTokens": max_tokens,
            "temperature": temperature
        }
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        return response.json(), None
    except Exception as e:
        return None, str(e)

print("✅ Helper function ready")


✅ Helper function ready


In [5]:
# Test 1: Claude Sonnet 4.5 - Simple test
sonnet_prompt = "Say 'Claude Sonnet 4.5 is working!' and explain what you're best at."

print("Testing Claude Sonnet 4.5...")
print(f"\nPrompt: {sonnet_prompt}")
print("\n" + "="*70)

if aws_bearer_token:
    try:
        response, error = call_bedrock_converse(
            sonnet_model_id, 
            sonnet_prompt, 
            max_tokens=500,
            temperature=0.7
        )
        
        if error:
            print(f"❌ Error: {error}")
        else:
            print(f"\n✅ Response structure:")
            print(f"Type: {type(response)}")
            print(f"Keys: {list(response.keys()) if isinstance(response, dict) else 'Not a dict'}")
            
            # Extract the text content from the response
            if 'output' in response:
                print(f"\nOutput keys: {list(response['output'].keys())}")
                if 'message' in response['output']:
                    print(f"Message keys: {list(response['output']['message'].keys())}")
                    if 'content' in response['output']['message']:
                        content_list = response['output']['message']['content']
                        print(f"Content type: {type(content_list)}")
                        print(f"Content length: {len(content_list)}")
                        if content_list and len(content_list) > 0:
                            first_content = content_list[0]
                            print(f"First content type: {type(first_content)}")
                            print(f"First content keys: {list(first_content.keys()) if isinstance(first_content, dict) else 'Not a dict'}")
                            
                            response_text = first_content.get('text', '') if isinstance(first_content, dict) else str(first_content)
                            print(f"\n✅ Response text:")
                            print(response_text)
                        else:
                            print(f"\n❌ Content list is empty")
                    else:
                        print(f"\n❌ No 'content' in message")
                else:
                    print(f"\n❌ No 'message' in output")
            else:
                print(f"\n❌ No 'output' in response")
                print(f"Full response: {response}")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  Bearer token not found")


Testing Claude Sonnet 4.5...

Prompt: Say 'Claude Sonnet 4.5 is working!' and explain what you're best at.


✅ Response structure:
Type: <class 'dict'>
Keys: ['metrics', 'output', 'stopReason', 'usage']

Output keys: ['message']
Message keys: ['content', 'role']
Content type: <class 'list'>
Content length: 1
First content type: <class 'dict'>
First content keys: ['text']

✅ Response text:
Claude Sonnet 4.5 is working!

I'm best at:

**Deep analytical thinking** - I excel at breaking down complex problems, reasoning through multiple angles, and providing nuanced analysis across topics like philosophy, science, strategy, and policy.

**Clear communication** - I can explain difficult concepts accessibly, adapt my style to different audiences, and structure information logically.

**Creative and technical writing** - From crafting stories and marketing copy to writing code and technical documentation, I can handle diverse writing tasks with attention to tone and purpose.

**Coding assistan

In [6]:
# Test 2: Claude Haiku 4.5 - With system prompt (like agent)
haiku_prompt = """Extract semantic roles from this interaction using Frame Semantics.

Query: test query
Response: test response

Output JSON with ARG0 (agent), ARG1 (theme), ARGM-TMP (temporal), ARGM-LOC (location)."""

print("Testing Claude Haiku 4.5...")
print(f"\nPrompt length: {len(haiku_prompt)} chars")
print("\n" + "="*70)

if aws_bearer_token:
    try:
        response, error = call_bedrock_converse(
            haiku_model_id, 
            haiku_prompt, 
            max_tokens=300,
            temperature=0.3
        )
        
        if error:
            print(f"❌ Error: {error}")
        else:
            print(f"\n✅ Response structure:")
            print(f"Keys: {list(response.keys())}")
            
            response_text = response['output']['message']['content'][0]['text']
            print("\n✅ Response text:")
            print(response_text)
            
            # Try to parse JSON
            try:
                cleaned = response_text.strip()
                if cleaned.startswith("```json"):
                    cleaned = cleaned[7:]
                elif cleaned.startswith("```"):
                    cleaned = cleaned[3:]
                if cleaned.endswith("```"):
                    cleaned = cleaned[:-3]
                cleaned = cleaned.strip()
                
                parsed = json.loads(cleaned)
                print(f"\n✅ JSON parsed: {parsed}")
            except json.JSONDecodeError as je:
                print(f"\n⚠️  JSON parse failed: {je}")
                print(f"Error at position: {je.pos if hasattr(je, 'pos') else 'unknown'}")
                print(f"Cleaned content (first 500 chars): {cleaned[:500]}")
                if hasattr(je, 'pos') and je.pos:
                    start = max(0, je.pos - 50)
                    end = min(len(cleaned), je.pos + 50)
                    print(f"Cleaned content (around error): {cleaned[start:end]}")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  Bearer token not found")


Testing Claude Haiku 4.5...

Prompt length: 198 chars


✅ Response structure:
Keys: ['metrics', 'output', 'stopReason', 'usage']

✅ Response text:
```json
{
  "query": {
    "text": "test query",
    "semantic_roles": {
      "ARG0": null,
      "ARG1": null,
      "ARGM-TMP": null,
      "ARGM-LOC": null
    },
    "note": "Insufficient semantic content for role extraction"
  },
  "response": {
    "text": "test response",
    "semantic_roles": {
      "ARG0": null,
      "ARG1": null,
      "ARGM-TMP": null,
      "ARGM-LOC": null
    },
    "note": "Insufficient semantic content for role extraction"
  }
}
```

**Explanation:** The provided query and response are placeholder texts without meaningful semantic content. To extract Frame Semantic roles, the text would need to contain:
- **ARG0 (Agent)**: An entity performing an action
- **ARG1 (Theme)**: The primary object/entity affected
- **ARGM-TMP (Temporal)**: Time expressions
- **ARGM-LOC (Location)**: Spatial references

Please pr